In [1]:
# this script replaces all entries for 2025Q4 and User google with updated google data for 2025Q4


import pandas as pd
import os

input1 = "../../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv"
input2 = "../../../50 KM Group/Royalties/Statements/Karen/Rock Music/Quarterly statements/2026 Q1/as sent by Rock/2025Q4 amended statements (20260718)/MOK 2025Q4 NewMeida_TWN_NEW_googleonly.xlsx"
outputfilename = "../../../50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_2021Q1_2026Q1_1_raw_combined_update.csv"

AMOUNT_COL = "SHARE AMOUNT (local FX)"
USER_COL = "USER"

# ---------- Load ----------
df1 = pd.read_csv(input1, low_memory=False)
df2 = pd.read_excel(input2)

# ---------- Helpers ----------
def amount(df):
    """Numeric sum of the share amount column (handles thousands separators)."""
    s = df[AMOUNT_COL]
    if s.dtype == object:
        s = s.astype(str).str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce").sum()

YEAR_COL = "Rev Year"
QUARTER_COL = "Rev Quarter"

def google_2025q4_mask(df):
    """Mask: User == Google AND Rev Year == 2025 AND Rev Quarter == Q4."""
    is_google = df[USER_COL].astype(str).str.strip().str.lower() == "google"
    is_2025 = df[YEAR_COL].astype(str).str.strip().str.replace(".0", "", regex=False) == "2025"
    is_q4 = df[QUARTER_COL].astype(str).str.strip().str.upper() == "Q4"
    return is_google & is_2025 & is_q4

# ---------- Checks BEFORE ----------
print("=== BEFORE ===")
print(f"input1 rows: {len(df1):,}")
print(f"input1 total {AMOUNT_COL}: {amount(df1):,.2f}")
g_before = df1[google_2025q4_mask(df1)]
print(f"input1 Google 2025 Q4 rows: {len(g_before):,}, {AMOUNT_COL}: {amount(g_before):,.2f}")
print(f"input2 rows: {len(df2):,}")
print(f"input2 total {AMOUNT_COL}: {amount(df2):,.2f}\n")

# ---------- Column consistency check ----------
missing_in_2 = set(df1.columns) - set(df2.columns)
extra_in_2 = set(df2.columns) - set(df1.columns)
if missing_in_2:
    print(f"WARNING - columns in input1 missing from input2: {sorted(missing_in_2)}")
if extra_in_2:
    print(f"WARNING - columns in input2 not in input1 (will be dropped): {sorted(extra_in_2)}")

# ---------- a) Delete only Google rows for Rev Year 2025 / Rev Quarter Q4 ----------
delete_mask = google_2025q4_mask(df1)
n_deleted = delete_mask.sum()
df1_clean = df1[~delete_mask]
print(f"\nDeleted {n_deleted:,} rows with User == 'Google' AND Rev Year == 2025 AND Rev Quarter == Q4")

# ---------- b) Append input2 ----------
df2_aligned = df2.reindex(columns=df1.columns)  # align to input1's column order
df_out = pd.concat([df1_clean, df2_aligned], ignore_index=True)

# ---------- Checks AFTER ----------
print("\n=== AFTER ===")
print(f"output rows: {len(df_out):,}  (= {len(df1):,} - {n_deleted:,} + {len(df2):,})")
print(f"output total {AMOUNT_COL}: {amount(df_out):,.2f}")
g_after = df_out[google_2025q4_mask(df_out)]
print(f"output Google 2025 Q4 rows: {len(g_after):,}, {AMOUNT_COL}: {amount(g_after):,.2f}")

# ---------- Save ----------
os.makedirs(os.path.dirname(outputfilename), exist_ok=True)
df_out.to_csv(outputfilename, index=False)
print(f"\nSaved: {outputfilename}")




=== BEFORE ===
input1 rows: 1,398,905
input1 total SHARE AMOUNT (local FX): 6,877,194.97
input1 Google 2025 Q4 rows: 1,008, SHARE AMOUNT (local FX): 68,138.88
input2 rows: 44,855
input2 total SHARE AMOUNT (local FX): 84,670.27

WARNING - columns in input1 missing from input2: ['Base Price', 'Release Date', 'Type', 'WS Price']

Deleted 1,008 rows with User == 'Google' AND Rev Year == 2025 AND Rev Quarter == Q4

=== AFTER ===
output rows: 1,442,752  (= 1,398,905 - 1,008 + 44,855)
output total SHARE AMOUNT (local FX): 6,893,726.36
output Google 2025 Q4 rows: 44,855, SHARE AMOUNT (local FX): 84,670.27

Saved: ../../../50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_2021Q1_2026Q1_1_raw_combined_update.csv


In [2]:
# Find common columns
common_cols = df_1c.columns.intersection(df_2c.columns)

# Dictionary to hold comparison results
type_comparison = {}

# Loop through each common column
for col in common_cols:
    # Get the set of types present in each column (ignoring NaN)
    types_df1 = set(df_1c[col].dropna().map(type))
    types_df2 = set(df_2c[col].dropna().map(type))
    
    # Store in the results dictionary
    type_comparison[col] = {
        "df1_types": types_df1,
        "df2_types": types_df2,
        "types_match": types_df1 == types_df2
    }

# Convert to a DataFrame for nicer display
type_comparison_df = pd.DataFrame(type_comparison).T

print(type_comparison_df)


NameError: name 'df_1c' is not defined

In [ ]:
df_1c, df_2c = align_columns(df_1c, df_2c)
df_final = pd.concat([df_1c, df_2c], ignore_index=True)
print(f"Total fee: {df_final['SHARE AMOUNT (local FX)'].sum()}. Total units: {df_final['UNIT'].sum()}")

df_final = df_final.sort_index(axis=1)

print('columns in df_final:')
for column in df_final.columns:        
        print(column)

print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")



Total fee: 6877188.30348313. Total units: 7353499212
columns in df_final:
AMOUNT
ARTIST
Base Price
CATALOG NO.
CATALOG NO._MOD
CATALOG TITLE
Ctrl.%
Currency
Entry No.
PayType
Payee/Licensor
Payer/Licensee
REVENUE PERIOD
ROYALTY
Release Date
Report Quarter
Report year
Rev Quarter
Rev Year
Royalty Rate%
SHARE AMOUNT (local FX)
SONG TITLE
Share%
SongProRata
Territory
Type
UNIT
USER
WS Price
Merged DataFrame: Rows: 1398905, Columns: 29
Merged DataFrame: Rows: 1398905, Columns: 29


In [ ]:
for col in df_final.columns:
    print(f"Column: {col}")
    print(df_final[col].map(type).value_counts())
    print()

Column: AMOUNT
AMOUNT
<class 'float'>    1398905
Name: count, dtype: int64

Column: ARTIST
ARTIST
<class 'str'>    1398905
Name: count, dtype: int64

Column: Base Price
Base Price
<class 'float'>    1398905
Name: count, dtype: int64

Column: CATALOG NO.
CATALOG NO.
<class 'str'>    1398905
Name: count, dtype: int64

Column: CATALOG NO._MOD
CATALOG NO._MOD
<class 'str'>    1398905
Name: count, dtype: int64

Column: CATALOG TITLE
CATALOG TITLE
<class 'str'>    1398905
Name: count, dtype: int64

Column: Ctrl.%
Ctrl.%
<class 'int'>    1398905
Name: count, dtype: int64

Column: Currency
Currency
<class 'str'>    1398905
Name: count, dtype: int64

Column: Entry No.
Entry No.
<class 'float'>    1398905
Name: count, dtype: int64

Column: PayType
PayType
<class 'str'>    1398905
Name: count, dtype: int64

Column: Payee/Licensor
Payee/Licensor
<class 'str'>    1398905
Name: count, dtype: int64

Column: Payer/Licensee
Payer/Licensee
<class 'str'>    1398905
Name: count, dtype: int64

Column: REVE

In [ ]:
df_final.to_csv(outputfilename, index=False)